<a href="https://colab.research.google.com/github/Janpu-Hou/Green-Learning-Basic/blob/main/Basic4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from scipy.special import softmax
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight # Added for class weights
import os # Added for file path checks

# =====================================================================
# MODULE 1 & 2: SAAB TRANSFORM & DFT (PRE-PROCESSING BLOCKS)
# =====================================================================
class TabularSaabTransform(BaseEstimator, TransformerMixin):
    def __init__(self, k_prime=0.95):
        self.k_prime = k_prime
        self.scaler = StandardScaler()
        self.pca = None
        self.bias_vector_ = None

    def fit(self, X, y=None):
        X_scaled = self.scaler.fit_transform(X)
        self.pca = PCA(n_components=self.k_prime, random_state=42)
        self.pca.fit(X_scaled)
        raw_responses = np.dot(X_scaled, self.pca.components_.T)
        self.bias_vector_ = np.abs(np.min(raw_responses, axis=0)) + 1e-5
        return self

    def transform(self, X):
        X_scaled = self.scaler.transform(X)
        ac_responses = np.dot(X_scaled, self.pca.components_.T)
        return ac_responses + self.bias_vector_

class MultiClassDiscriminantFeatureTest(BaseEstimator, TransformerMixin):
    def __init__(self, percentile_threshold=70):
        self.percentile_threshold = percentile_threshold
        self.selected_indices_ = []

    def _calculate_entropy(self, y):
        if len(y) == 0: return 0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs + 1e-9))

    def fit(self, X, y):
        num_features = X.shape[1]
        feature_costs = []
        y = np.array(y)

        for i in range(num_features):
            f_col = X[:, i]
            thresholds = np.percentile(f_col, [25, 50, 75])
            best_entropy = float('inf')

            for t in thresholds:
                left_mask = f_col < t
                if np.sum(left_mask) == 0 or np.sum(~left_mask) == 0:
                    continue
                entropy_split = (
                    (np.sum(left_mask) / len(y)) * self._calculate_entropy(y[left_mask]) +
                    (np.sum(~left_mask) / len(y)) * self._calculate_entropy(y[~left_mask])
                )
                if entropy_split < best_entropy:
                    best_entropy = entropy_split
            feature_costs.append(best_entropy if best_entropy != float('inf') else 1.0)

        cost_cutoff = np.percentile(feature_costs, self.percentile_threshold)
        self.selected_indices_ = [idx for idx, cost in enumerate(feature_costs) if cost <= cost_cutoff]
        return self

    def transform(self, X):
        return X[:, self.selected_indices_]

# =====================================================================
# MODIFIED REGRESSION SLM TREE NODE FOR RESIDUAL FITTING (SLR)
# =====================================================================
class SLMRegTreeNode:
    def __init__(self, depth=0, max_depth=4):
        self.depth = depth
        self.max_depth = max_depth
        self.w = None          # Hyperplane projection weights
        self.threshold = None  # Scalar split point
        self.left = None       # Left child
        self.right = None      # Right child
        self.is_leaf = False
        self.value = None      # Leaf regression prediction output value

class SLMRegTree:
    """Subspace Learning Regressor (SLR) Tree to fit raw floating-point gradients."""
    def __init__(self, max_depth=4, num_proj_candidates=20, random_state=42):
        self.max_depth = max_depth
        self.num_proj_candidates = num_proj_candidates
        self.random_state = random_state
        self.root = None

    def _find_best_oblique_split(self, X, residuals, rng):
        num_samples, num_features = X.shape
        best_mse_reduction = -1.0
        best_w, best_t = None, None

        initial_mse = np.var(residuals)

        candidates = []
        for i in range(min(num_features, self.num_proj_candidates // 2)):
            w = np.zeros(num_features)
            w[i] = 1.0
            candidates.append(w)

        while len(candidates) < self.num_proj_candidates:
            w = rng.normal(0, 1, size=num_features)
            norm = np.linalg.norm(w)
            if norm > 0:
                candidates.append(w / norm)

        for w in candidates:
            X_projected = np.dot(X, w)
            split_targets = np.percentile(X_projected, [20, 40, 60, 80])

            for t in split_targets:
                left_mask = X_projected < t
                right_mask = ~left_mask

                if np.sum(left_mask) < 2 or np.sum(right_mask) < 2:
                    continue

                mse_split = (np.sum(left_mask) * np.var(residuals[left_mask]) +
                             np.sum(right_mask) * np.var(residuals[right_mask])) / num_samples

                mse_reduction = initial_mse - mse_split
                if mse_reduction > best_mse_reduction:
                    best_mse_reduction = mse_reduction
                    best_w = w
                    best_t = t

        return best_w, best_t

    def _build_tree(self, X, residuals, node, rng):
        num_samples = X.shape[0]
        node.value = np.mean(residuals) if num_samples > 0 else 0.0

        if node.depth >= node.max_depth or num_samples < 5:
            node.is_leaf = True
            return

        w, t = self._find_best_oblique_split(X, residuals, rng)
        if w is None or t is None:
            node.is_leaf = True
            return

        node.w = w
        node.threshold = t

        X_projected = np.dot(X, w)
        left_mask = X_projected < t

        if np.sum(left_mask) == 0 or np.sum(~left_mask) == 0:
            node.is_leaf = True
            return

        node.left = SLMRegTreeNode(depth=node.depth + 1, max_depth=node.max_depth)
        node.right = SLMRegTreeNode(depth=node.depth + 1, max_depth=node.max_depth)

        self._build_tree(X[left_mask], residuals[left_mask], node.left, rng)
        self._build_tree(X[~left_mask], residuals[~left_mask], node.right, rng)

    def fit(self, X, residuals):
        rng = np.random.default_rng(self.random_state)
        self.root = SLMRegTreeNode(depth=0, max_depth=self.max_depth)
        self._build_tree(X, residuals, self.root, rng)
        return self

    def _predict_sample(self, x, node):
        if node.is_leaf:
            return node.value
        if np.dot(x, node.w) < node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)

    def predict(self, X):
        return np.array([self._predict_sample(x, self.root) for x in X])


# =====================================================================
# SYSTEM ARCHITECTURE: SLM BOOST CLASS FIRE
# =====================================================================
class SLMBoostClassifier(BaseEstimator, ClassifierMixin):
    """
    SLM Boost Multi-Class Classifier.
    Sequentially builds ensembles of Subspace Learning Regressor (SLR)
    Trees to minimize Multi-Class Log-Loss.
    """
    def __init__(self, n_estimators=15, learning_rate=0.1, max_depth=4, num_proj_candidates=20, random_state=42, class_weight=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.num_proj_candidates = num_proj_candidates
        self.random_state = random_state
        self.class_weight = class_weight # New parameter for class weighting
        self.estimators_ = []  # Matrix structured storage layout: [round][class_idx]
        self.classes_ = None
        self.class_weights_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        num_classes = len(self.classes_)
        num_samples = X.shape[0]

        # Calculate class weights if specified
        if self.class_weight is not None:
            class_weights_dict = compute_class_weight(
                class_weight=self.class_weight,
                classes=self.classes_,
                y=y
            )
            # Map class weights to an array ordered by self.classes_
            self.class_weights_ = np.array([class_weights_dict[c] for c in self.classes_])
        else:
            self.class_weights_ = np.ones(num_classes) # No weighting

        # 1. Initialize Log-odds margins to 0.0
        F = np.zeros((num_samples, num_classes))

        # Convert target vectors to multi-class one-hot matrices
        y_onehot = np.zeros((num_samples, num_classes))
        for i, c in enumerate(self.classes_):
            y_onehot[:, i] = (y == c).astype(int)

        print(f" -> Beginning Boosting Execution Loop iterations...")
        for r in range(self.n_estimators):
            # Compute probability metrics mapping across the samples via softmax
            probs = softmax(F, axis=1)

            # Step 2: Compute pseudo-residuals (Gradients = True Labels - Softmax Probabilities)
            residuals = y_onehot - probs

            round_estimators = []
            # Fit one oblique SLR tree for each discrete category gradient
            for c_idx in range(num_classes):
                # Apply class weight to the residuals for the current class
                weighted_residuals = residuals[:, c_idx] * self.class_weights_[c_idx]

                tree = SLMRegTree(
                    max_depth=self.max_depth,
                    num_proj_candidates=self.num_proj_candidates,
                    random_state=self.random_state + r + c_idx
                )
                tree.fit(X, weighted_residuals) # Fit with weighted residuals

                # Update current raw margin predictions using calculated step updates
                F[:, c_idx] += self.learning_rate * tree.predict(X)
                round_estimators.append(tree)

            self.estimators_.append(round_estimators)
            if (r + 1) % 5 == 0 or r == 0:
                print(f"    Round {r+1}/{self.n_estimators} Complete.")
        return self

    def predict_proba(self, X):
        num_samples = X.shape[0]
        num_classes = len(self.classes_)
        F = np.zeros((num_samples, num_classes))

        # Score mapping loops across ensemble matrices sequential rounds
        for round_estimators in self.estimators_:
            for c_idx, tree in enumerate(round_estimators):
                F[:, c_idx] += self.learning_rate * tree.predict(X)

        return softmax(F, axis=1)

    def predict(self, X):
      probas = self.predict_proba(X)
      return self.classes_[np.argmax(probas, axis=1)]

# =======================================================
# INTEGRATED UNSW-NB15 SLM BOOST PIPELINE PIPELINE
# =======================================================

def run_slm_boost_pipeline(train_path, test_path):
  print("Step 0: Processing Tabular Datasets & Multi-Class Categories...")

  # Download datasets if they don't exist
  if not os.path.exists(train_path):
      print(f"Downloading {train_path}...")
      !wget -q -O {train_path} https://raw.githubusercontent.com/Anirudh257/UNSW-NB15-Dataset-for-Attack-Classification/main/UNSW_NB15_training-set.csv
  if not os.path.exists(test_path):
      print(f"Downloading {test_path}...")
      !wget -q -O {test_path} https://raw.githubusercontent.com/Anirudh257/UNSW-NB15-Dataset-for-Attack-Classification/main/UNSW_NB15_testing-set.csv

  train_df = pd.read_csv(train_path)
  test_df = pd.read_csv(test_path)

  for df in [train_df, test_df]:
    if 'attack_cat' in df.columns:
      df['attack_cat'] = df['attack_cat'].str.strip().fillna('Normal')
      df.loc[df['attack_cat'] == '', 'attack_cat'] = 'Normal'

  label_encoder = LabelEncoder()
  y_train = label_encoder.fit_transform(train_df['attack_cat'])
  y_test = label_encoder.transform(test_df['attack_cat'])

  cols_to_drop = ['id', 'label', 'attack_cat']
  X_train_raw = train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns])
  X_test_raw = test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns])

  categorical_cols = X_train_raw.select_dtypes(include=['object']).columns.tolist()
  X_combined = pd.concat([X_train_raw, X_test_raw], axis=0)
  X_combined_encoded = pd.get_dummies(X_combined, columns=categorical_cols, drop_first=True)

  X_train_numeric = X_combined_encoded.iloc[:len(X_train_raw)].values.astype(np.float64)
  X_test_numeric = X_combined_encoded.iloc[len(X_train_raw):].values.astype(np.float64)

  # --- Module 1: Unsupervised Representation learning ---
  print("\nStep 1: Commencing Tabular Saab Subspace Approximation...")
  saab = TabularSaabTransform(k_prime=0.95)
  X_train_rep = saab.fit_transform(X_train_numeric)
  X_test_rep = saab.transform(X_test_numeric)

  # --- Module 2: Supervised Feature selection ---
  print("\nStep 2: Executing Multi-Class Discriminant Feature Test (DFT)...")
  dft = MultiClassDiscriminantFeatureTest(percentile_threshold=70)
  X_train_features = dft.fit_transform(X_train_rep, y_train)
  X_test_features = dft.transform(X_test_rep)

  # --- Step 2.5: Applying SMOTE to address class imbalance ---
  print("\nStep 2.5: Applying SMOTE to address class imbalance...")
  smote = SMOTE(random_state=42)
  X_train_features_resampled, y_train_resampled = smote.fit_resample(X_train_features, y_train)

  print(f"Original training samples: {X_train_features.shape[0]}")
  print(f"Resampled training samples: {X_train_features_resampled.shape[0]}")
  print("Class distribution after SMOTE:")
  unique, counts = np.unique(y_train_resampled, return_counts=True)
  for label, count in zip(label_encoder.inverse_transform(unique), counts):
      print(f"- {label}: {count}")

  # Update the training data used for the SLMBoostClassifier
  X_train_features = X_train_features_resampled
  y_train = y_train_resampled

  # --- Module 3: Custom SLM Boost Classifier Execution ---
  print("\nStep 3: Training Custom Subspace Learning Machine (SLM) Boost Pipeline Architecture...")
  slm_boost = SLMBoostClassifier(n_estimators=15, learning_rate=0.1, max_depth=4, num_proj_candidates=30, random_state=42, class_weight='balanced') # Added class_weight
  slm_boost.fit(X_train_features, y_train)

  predictions = slm_boost.predict(X_test_features)
  print("\n================ SLM BOOST EVALUATION REPORT ================")
  print(f"Oblique SLM Boost Testing Pipeline Accuracy: {accuracy_score(y_test, predictions):.4f}")
  print(classification_report(y_test, predictions, target_names=label_encoder.classes_, zero_division=0))

  print("\n================ THRESHOLD ADJUSTMENT FOR MINORITY CLASSES ================")
  # Get probabilities
  probas = slm_boost.predict_proba(X_test_features)

  # Identify a minority class with low precision to target, e.g., 'Worms'
  # Find its index in label_encoder.classes_
  target_class_name = 'Worms' # Or choose another class like 'Shellcode', 'Backdoor'
  try:
      target_class_idx = np.where(label_encoder.classes_ == target_class_name)[0][0]
      print(f"Targeting class '{target_class_name}' (index {target_class_idx}) for threshold adjustment.")
  except IndexError:
      print(f"Class '{target_class_name}' not found in target_names. Skipping threshold adjustment.")
      target_class_idx = None

  if target_class_idx is not None:
      # Define a range of thresholds to test for the target class
      # Start from a low threshold, as precision is very low, to explore the trade-off
      thresholds_to_test = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

      for threshold_val in thresholds_to_test:
          print(f"\n--- Applying threshold {threshold_val:.2f} for '{target_class_name}' ---")
          adjusted_predictions = np.zeros_like(y_test) # Initialize with placeholder

          for i in range(len(y_test)):
              if probas[i, target_class_idx] >= threshold_val:
                  # If probability for target class is above threshold, predict it
                  adjusted_predictions[i] = target_class_idx
              else:
                  # Otherwise, find the next most probable class (excluding the target class)
                  temp_probas = np.copy(probas[i, :])
                  temp_probas[target_class_idx] = -np.inf # Exclude target class from argmax
                  adjusted_predictions[i] = np.argmax(temp_probas)

          # Evaluate with adjusted predictions
          print(f"Oblique SLM Boost Testing Pipeline Accuracy (Threshold {threshold_val:.2f} for {target_class_name}): {accuracy_score(y_test, adjusted_predictions):.4f}")
          print(classification_report(y_test, adjusted_predictions, target_names=label_encoder.classes_, zero_division=0))


if __name__ == "__main__":
    run_slm_boost_pipeline("UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv")
    pass

Step 0: Processing Tabular Datasets & Multi-Class Categories...

Step 1: Commencing Tabular Saab Subspace Approximation...

Step 2: Executing Multi-Class Discriminant Feature Test (DFT)...

Step 2.5: Applying SMOTE to address class imbalance...
Original training samples: 175341
Resampled training samples: 560000
Class distribution after SMOTE:
- Analysis: 56000
- Backdoor: 56000
- DoS: 56000
- Exploits: 56000
- Fuzzers: 56000
- Generic: 56000
- Normal: 56000
- Reconnaissance: 56000
- Shellcode: 56000
- Worms: 56000

Step 3: Training Custom Subspace Learning Machine (SLM) Boost Pipeline Architecture...
 -> Beginning Boosting Execution Loop iterations...
    Round 1/15 Complete.
    Round 5/15 Complete.
    Round 10/15 Complete.
    Round 15/15 Complete.

================ SLM BOOST EVALUATION REPORT ================
Oblique SLM Boost Testing Pipeline Accuracy: 0.5605
                precision    recall  f1-score   support

      Analysis       0.02      0.07      0.03       677
      Bac